# DBMI AI Workshop (TypeScript edition)

Welcome! In this notebook, you will learn how to talk to AI models using **TypeScript**, running on **Deno**. We will use a library called **OpenAI Agents** made by OpenAI.

Even though the Agents SDK comes from OpenAI, it can also be used with models from other companies like Google or Anthropic. For this workshop, we'll stick with OpenAI's models to keep things simple.

Run each cell in order. If you get stuck, just ask an instructor!

## 0. One-time setup for Google Colab

Colab doesn't ship with the Deno runtime, so the first cell below installs it and registers Deno as a Jupyter kernel.

**After the cell finishes:**
1. In the Colab menu, click **Runtime → Change runtime type**.
2. Pick **Deno** from the kernel list and click **Save**.
3. Continue from section 1 below.

You only need to do this once per Colab session.

In [ ]:
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh -s -- -y
!deno jupyter --install
print("Deno kernel installed. Now switch the runtime: Runtime → Change runtime type → Deno.")

## 1. Import the tools

In Deno, we don't have to install anything — packages are loaded straight from npm by URL the first time the cell runs. The cell below pulls in everything we need so TypeScript understands how to build and run AI assistants.

A "Play" button should appear when you hover over the code below. Press the "Play" button to run the code.

In [ ]:
import {
  Agent,
  OpenAIProvider,
  run,
  setDefaultModelProvider,
  setTracingDisabled,
  tool,
} from "npm:@openai/agents@0.11.5";
import { z } from "npm:zod@4.4.3";

console.log("OpenAI Agents SDK ready.");

## 2. Get your personal Workshop Key

**Why this exists:** Think of an **'API Key'** like a **password**.

Whenever you send a request to an AI provider like OpenAI, Anthropic, Google, or AWS, you must provide this key. It is what identifies you to their systems and allows them to track your usage and billing.

To keep things simple, we have created a special "Workshop Key" for you to use today. This key acts as your password to access the models without needing to set up your own personal accounts or share credit card info.

**What to do:**
1. Visit [https://dbmi-ai-workshop.dbmi.deno.net/register](https://dbmi-ai-workshop.dbmi.deno.net/register)
2. Enter your email and the code we give you in class. (The code is: DBMI-WORKSHOP-MAY26)
3. **Copy the Workshop Key** that appears on the screen.
4. **Run the code cell below** (click the Play button).
5. **Paste your key** into the text box that appears and press **Enter**.

In [ ]:
const workshopKey = prompt(
  "Paste your Workshop Key (from https://dbmi-ai-workshop.dbmi.deno.net/register):",
)?.trim();

if (!workshopKey) {
  throw new Error("No Workshop Key provided.");
}

Deno.env.set("OPENAI_API_KEY", workshopKey);
console.log("Workshop Key saved for this session.");

## 3. Connect to the AI service

This cell tells TypeScript to send your requests through our workshop server. The server checks your key and passes your message to OpenAI.

**Keep your key safe:**
1. **Treat it like a password.** If someone else has it, they can use up your balance.
2. **Don't share it** in screenshots or messages.
3. **Safety first:** Our workshop setup makes it easy to reset an API key and limit costs. Each key starts with a small **\$10 limit**. You may be surprised how far that can go if you use the 'nano' and 'mini' models!

In [ ]:
setDefaultModelProvider(
  new OpenAIProvider({
    apiKey: Deno.env.get("OPENAI_API_KEY"),
    baseURL: "https://dbmi-ai-workshop.dbmi.deno.net/v1",
  }),
);
setTracingDisabled(true);

console.log("Connected to the workshop server.");

## 4. Build your first AI Agent

An **Agent** is like a tiny digital assistant. You give it a name, a personality (instructions), and choose which model it should use.

AI cost is measured in **tokens** (about 4 characters each). "Input" is what you send the model; "Output" is the reply it thinks of and sends back.

| Model | Cost | Input (\$ per 1M tokens) | Output (\$ per 1M tokens) | Best for... |
|---|---|---|---|---|
| `gpt-5.4-nano` | Cheapest | \$0.20 | \$1.25 | Simple tasks, very fast. |
| `gpt-5.4-mini` | Medium | \$0.75 | \$4.50 | A balanced default; smarter than nano. |
| `gpt-5.5` | Expensive | \$5.00 | \$30.00 | Very complex logic (uses budget fast). |

**Real-world example:** Sending an entire 50,000-word medical chart (\~70,000 input tokens) and getting back a summary (\~1,000 output tokens) would cost roughly:

- **gpt-5.4-nano:** \~1.5¢
- **gpt-5.4-mini:** \~6¢
- **gpt-5.5:** \~38¢

**Pick the smallest model that works for your task!** Your \$10 budget lasts for hundreds of summaries on 'nano', but only about 26 on the biggest model.

In [ ]:
const agent = new Agent({
  name: "Assistant",
  instructions: "You are a helpful assistant.",
  model: "gpt-5.4-mini",
});

const result = await run(agent, "Write a haiku about a great AI workshop.");
console.log(result.finalOutput);

## 5. Give your agent a Tool

AI models are smart, but they are frozen in time—they don't know what's happening in the world *right now*. If you ask about the weather or a current news story, a standard AI has to guess or admit it doesn't know.

You can give an agent a **tool**, which allows it to look up live information. In this example, we'll give one agent a tool to 'check the weather'.

In [ ]:
// 1. Define the Tool with Salt Lake City hardcoded
const getWeather = tool({
  name: "get_weather",
  description: "Get the current weather for Salt Lake City using the wttr.in service.",
  parameters: z.object({}),
  execute: async () => {
    try {
      const response = await fetch("https://wttr.in/Salt+Lake+City?format=3");
      return (await response.text()).trim();
    } catch {
      return "Could not find weather for Salt Lake City.";
    }
  },
});

// 2. Setup an agent WITHOUT the tool
const basicAgent = new Agent({
  name: "Basic Assistant",
  instructions: "You are a helpful assistant.",
  model: "gpt-5.4-mini",
});

// 3. Setup an agent WITH the tool
const toolEnabledAgent = new Agent({
  name: "Tool Assistant",
  instructions: "You are a helpful assistant. Use your tools to check live info.",
  model: "gpt-5.4-mini",
  tools: [getWeather],
});

console.log("--- ASKING BASIC AGENT ---");
const res1 = await run(basicAgent, "What is the weather in Salt Lake City right now?");
console.log(res1.finalOutput);

console.log("\n--- ASKING AGENT WITH TOOL ---");
const res2 = await run(toolEnabledAgent, "What is the weather in Salt Lake City right now?");
console.log(res2.finalOutput);

## 6. Build an AI Team

For big projects, one AI might get confused if you ask it to do too much at once. Instead, you can build a **team of agents** where each one has a specific job. Think of it like an assembly line:

1. **The Writer** creates a basic story.
2. **The Poet** takes that story and makes it rhyme.
3. **The Translator** takes the poem and changes the language.

By passing the work from one agent to the next, you may get much better results!

In [ ]:
const storyWriter = new Agent({
  name: "Story Writer",
  instructions: "Write a very short story (3-4 sentences) on the given topic.",
  model: "gpt-5.4-mini",
});

const poet = new Agent({
  name: "Poet",
  instructions: "Rewrite the given text as a short rhyming poem (4-8 lines).",
  model: "gpt-5.4-mini",
});

const pigLatinTranslator = new Agent({
  name: "Pig Latin Translator",
  instructions:
    "Translate the given text into pig latin. " +
    "Move the first consonant cluster to the end and add 'ay'. " +
    "Words starting with a vowel get 'way' appended. " +
    "Preserve line breaks.",
  model: "gpt-5.4-mini",
});

// Step 1 — write the story
const story = (await run(storyWriter, "a fascinating AI workshop being taken in Salt Lake City")).finalOutput;
console.log("STORY:\n" + story + "\n");

// Step 2 — turn the story into a poem
const poem = (await run(poet, story ?? "")).finalOutput;
console.log("POEM:\n" + poem + "\n");

// Step 3 — translate the poem into pig latin
const pigLatin = (await run(pigLatinTranslator, poem ?? "")).finalOutput;
console.log("PIG LATIN:\n" + pigLatin);

## 7. Challenge: The Medical Summary Team

Now, let's put it all together. We will create a team to handle a medical discharge summary.

1. **The Simplifier:** Takes medical jargon and explains it in plain English.
2. **The Action Plan:** Extracts exactly what the patient needs to do (pills, appointments).
3. **The Translator:** Translates the summary into another language (e.g., Spanish).

In [ ]:
const medicalNote = `
Patient presents with acute exacerbation of chronic obstructive pulmonary disease (COPD).
Prescribed Prednisone 40mg daily for 5 days and Albuterol HFA inhaler q4h PRN
in addition to existing inhaler regimen.
Follow up with primary care in 1 week. Monitor for new fever or worsening symptoms.
Low threshold to add antibiotics for CAP.
`;

const simplifier = new Agent({
  name: "Simplifier",
  instructions: "Translate medical jargon into very simple, plain English.",
  model: "gpt-5.4-mini",
});

const actionPlanner = new Agent({
  name: "Action Planner",
  instructions: "Create a bulleted 'To-Do' list for the patient based on the summary.",
  model: "gpt-5.4-mini",
});

const translator = new Agent({
  name: "Spanish Translator",
  instructions: "Translate the given text into Spanish.",
  model: "gpt-5.4-mini",
});

// 1. Simplify
const simpleText = (await run(simplifier, medicalNote)).finalOutput;
console.log("--- SIMPLE SUMMARY ---");
console.log(simpleText);

// 2. Get the To-Do list
const toDo = (await run(actionPlanner, simpleText ?? "")).finalOutput;
console.log("\n--- PATIENT TO-DO LIST ---");
console.log(toDo);

// 3. Translate the list to Spanish
const spanishList = (await run(translator, toDo ?? "")).finalOutput;
console.log("\n--- LISTA DE TAREAS (SPANISH) ---");
console.log(spanishList);